# Software Component 2: Modelling Complex, Semi-Structured JSON Data using PySpark

## Installing PySpark

In [16]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


## Import Libraries

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

import os

# Implementation

## Measurements Data

### Create a Spark Session

In [377]:
# Create a Spark session
spark = SparkSession.builder \
    .appName("jsonFlatten")\
    .master("local[*]")\
    .getOrCreate()

### Load the JSON Data

In [4]:
# Load the JSON data
df = spark.read.format('json').load("raw_data/measurements_data/2026-01-16T09-00-00+00-00.json")

In [5]:
df.show()

+--------------------+--------------------+--------------------+
|            @context|               items|                meta|
+--------------------+--------------------+--------------------+
|http://environmen...|[{http://environm...|{Status: Beta ser...|
+--------------------+--------------------+--------------------+



In [6]:
df.printSchema()

root
 |-- @context: string (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- @id: string (nullable = true)
 |    |    |-- datumType: string (nullable = true)
 |    |    |-- label: string (nullable = true)
 |    |    |-- latestReading: struct (nullable = true)
 |    |    |    |-- @id: string (nullable = true)
 |    |    |    |-- date: string (nullable = true)
 |    |    |    |-- dateTime: string (nullable = true)
 |    |    |    |-- measure: string (nullable = true)
 |    |    |    |-- value: string (nullable = true)
 |    |    |-- notation: string (nullable = true)
 |    |    |-- parameter: string (nullable = true)
 |    |    |-- parameterName: string (nullable = true)
 |    |    |-- period: long (nullable = true)
 |    |    |-- qualifier: string (nullable = true)
 |    |    |-- station: string (nullable = true)
 |    |    |-- stationReference: string (nullable = true)
 |    |    |-- unit: string (nullable = true)
 |   

Based on the Schema above, it is shown that the actual data is located inside the "items" array. In PySpark, we will be using the .explore() command to flatten the array.

### Flattening & Normalising the Structure

In [342]:
# --- Flatten the nested JSON structure ---
# 1. Select the 'items' array and ignore 'context' and 'metadata' fields
measurements_df = df.select(explode("items").alias("item")) \
    .select(
        col("item.@id").alias("measurementId"),
        col("item.stationReference"),
        col("item.parameterName").alias("parameter"),
        col("item.period"),
        col("item.qualifier"),
        col("item.valueType"),
        col("item.unitName"),
        )

# Normalise latestReading struct to a separate Spark DataFrame
readings_df = df.select(explode("items").alias("item")) \
    .select(
        col("item.latestReading.@id").alias("readingId"),
        col("item.latestReading.dateTime").alias("readingDatetime"),
        col("item.latestReading.value").alias("readingValue"),
        col("item.latestReading.measure").alias("measurementId")
        )

In [343]:
measurements_df.show()

+--------------------+----------------+-----------+------+----------------+-------------+--------+
|       measurementId|stationReference|  parameter|period|       qualifier|    valueType|unitName|
+--------------------+----------------+-----------+------+----------------+-------------+--------+
|http://environmen...|          1029TH|Water Level|   900|Downstream Stage|instantaneous|    mASD|
|http://environmen...|          1029TH|Water Level|   900|           Stage|instantaneous|    mASD|
|http://environmen...|           E2043|Water Level|   900|           Stage|instantaneous|    mASD|
|http://environmen...|           52119|Water Level|   900|           Stage|instantaneous|    mASD|
|http://environmen...|          E21136|Water Level|   900|           Stage|instantaneous|    mASD|
|http://environmen...|            2067|Water Level|   900|           Stage|instantaneous|    mASD|
|http://environmen...|           48143|Water Level|   900|Downstream Stage|instantaneous|    mASD|
|http://en

In [344]:
readings_df.show()

+--------------------+--------------------+------------+--------------------+
|           readingId|     readingDatetime|readingValue|       measurementId|
+--------------------+--------------------+------------+--------------------+
|http://environmen...|2026-01-16T08:15:00Z|       0.159|http://environmen...|
|http://environmen...|2026-01-16T08:15:00Z|       0.371|http://environmen...|
|http://environmen...|2026-01-16T08:30:00Z|      -0.081|http://environmen...|
|http://environmen...|2026-01-16T08:30:00Z|        2.02|http://environmen...|
|http://environmen...|2026-01-16T08:15:00Z|       0.341|http://environmen...|
|http://environmen...|2026-01-16T08:30:00Z|       1.433|http://environmen...|
|http://environmen...|2026-01-16T04:00:00Z|     -13.012|http://environmen...|
|http://environmen...|2026-01-16T08:30:00Z|       0.392|http://environmen...|
|                NULL|                NULL|        NULL|                NULL|
|                NULL|                NULL|        NULL|        

### Cleaning the Data

Columns removed:
- notation (redundant to qualifier & stationReference)
- label (irrelevant as we will acquire context from Stations Data)
- datumType (irrelevant)
- readingDate (redundant as we will be transforming the readingDateTime into a 'timestamp' type)

In [345]:
for col_name in measurements_df.columns:
    null_values = measurements_df.where(col(col_name).isNull()).count()
    print(f"Null_values in '{col_name}': {null_values}")

Null_values in 'measurementId': 0
Null_values in 'stationReference': 0
Null_values in 'parameter': 0
Null_values in 'period': 14
Null_values in 'qualifier': 0
Null_values in 'valueType': 0
Null_values in 'unitName': 0


In [346]:
for col_name in readings_df.columns:
    null_values = readings_df.where(col(col_name).isNull()).count()
    print(f"Null_values in '{col_name}': {null_values}")

Null_values in 'readingId': 1521
Null_values in 'readingDatetime': 1521
Null_values in 'readingValue': 1522
Null_values in 'measurementId': 1521


Remove NULL rows in Readings DF

In [347]:
readings_df = readings_df.where(col("readingId").isNotNull()) 

In [348]:
readings_df.show()

+--------------------+--------------------+------------+--------------------+
|           readingId|     readingDatetime|readingValue|       measurementId|
+--------------------+--------------------+------------+--------------------+
|http://environmen...|2026-01-16T08:15:00Z|       0.159|http://environmen...|
|http://environmen...|2026-01-16T08:15:00Z|       0.371|http://environmen...|
|http://environmen...|2026-01-16T08:30:00Z|      -0.081|http://environmen...|
|http://environmen...|2026-01-16T08:30:00Z|        2.02|http://environmen...|
|http://environmen...|2026-01-16T08:15:00Z|       0.341|http://environmen...|
|http://environmen...|2026-01-16T08:30:00Z|       1.433|http://environmen...|
|http://environmen...|2026-01-16T04:00:00Z|     -13.012|http://environmen...|
|http://environmen...|2026-01-16T08:30:00Z|       0.392|http://environmen...|
|http://environmen...|2026-01-16T08:30:00Z|       0.592|http://environmen...|
|http://environmen...|2026-01-16T08:30:00Z|       0.504|http://e

Check for malformed data (in this case, values with a LIST of values)

In [349]:
for col_name in measurements_df.columns:
    malformed_count = measurements_df.filter(col(col_name).contains("[")).count()
    print(f"Malformed values in {col_name}: {malformed_count}")

Malformed values in measurementId: 0
Malformed values in stationReference: 0
Malformed values in parameter: 0
Malformed values in period: 0
Malformed values in qualifier: 0
Malformed values in valueType: 0
Malformed values in unitName: 0


In [350]:
for col_name in readings_df.columns:
    malformed_count = readings_df.filter(col(col_name).contains("[")).count()
    print(f"Malformed values in {col_name}: {malformed_count}")

Malformed values in readingId: 0
Malformed values in readingDatetime: 0
Malformed values in readingValue: 2
Malformed values in measurementId: 0


### Data Transformation

In [351]:
# Extract IDs from the URL fields
transformed_measurements_df = measurements_df.withColumn(
    "measurementId", regexp_extract(col("measurementId"), r'measures/(.+)$', 1) # extract the reading ID from the full URL
)
transformed_measurements_df.show()

+--------------------+----------------+-----------+------+----------------+-------------+--------+
|       measurementId|stationReference|  parameter|period|       qualifier|    valueType|unitName|
+--------------------+----------------+-----------+------+----------------+-------------+--------+
|1029TH-level-down...|          1029TH|Water Level|   900|Downstream Stage|instantaneous|    mASD|
|1029TH-level-stag...|          1029TH|Water Level|   900|           Stage|instantaneous|    mASD|
|E2043-level-stage...|           E2043|Water Level|   900|           Stage|instantaneous|    mASD|
|52119-level-stage...|           52119|Water Level|   900|           Stage|instantaneous|    mASD|
|E21136-level-stag...|          E21136|Water Level|   900|           Stage|instantaneous|    mASD|
|2067-level-stage-...|            2067|Water Level|   900|           Stage|instantaneous|    mASD|
|48143-level-downs...|           48143|Water Level|   900|Downstream Stage|instantaneous|    mASD|
|720215-le

In [352]:
# Extract IDs from the URL fields
transformed_readings_df = readings_df.withColumn(
    "readingId", regexp_extract(col("readingId"), r'readings/(.+)$', 1) # extract the reading ID from the full URL
).withColumn(
    "measurementId", regexp_extract(col("measurementId"), r'measures/(.+)$', 1) # extract the measures ID from the full URL
)
transformed_readings_df.show()

+--------------------+--------------------+------------+--------------------+
|           readingId|     readingDatetime|readingValue|       measurementId|
+--------------------+--------------------+------------+--------------------+
|1029TH-level-down...|2026-01-16T08:15:00Z|       0.159|1029TH-level-down...|
|1029TH-level-stag...|2026-01-16T08:15:00Z|       0.371|1029TH-level-stag...|
|E2043-level-stage...|2026-01-16T08:30:00Z|      -0.081|E2043-level-stage...|
|52119-level-stage...|2026-01-16T08:30:00Z|        2.02|52119-level-stage...|
|E21136-level-stag...|2026-01-16T08:15:00Z|       0.341|E21136-level-stag...|
|2067-level-stage-...|2026-01-16T08:30:00Z|       1.433|2067-level-stage-...|
|48143-level-downs...|2026-01-16T04:00:00Z|     -13.012|48143-level-downs...|
|720215-level-stag...|2026-01-16T08:30:00Z|       0.392|720215-level-stag...|
|F1906-level-stage...|2026-01-16T08:30:00Z|       0.592|F1906-level-stage...|
|E8266-level-stage...|2026-01-16T08:30:00Z|       0.504|E8266-le

In [340]:
transformed_readings_df.show()

+--------------------+--------------------+------------+--------------------+
|           readingId|     readingDatetime|readingValue|       measurementId|
+--------------------+--------------------+------------+--------------------+
|1029TH-level-down...|2026-01-16T08:15:00Z|       0.159|1029TH-level-down...|
|1029TH-level-stag...|2026-01-16T08:15:00Z|       0.371|1029TH-level-stag...|
|E2043-level-stage...|2026-01-16T08:30:00Z|      -0.081|E2043-level-stage...|
|52119-level-stage...|2026-01-16T08:30:00Z|        2.02|52119-level-stage...|
|E21136-level-stag...|2026-01-16T08:15:00Z|       0.341|E21136-level-stag...|
|2067-level-stage-...|2026-01-16T08:30:00Z|       1.433|2067-level-stage-...|
|48143-level-downs...|2026-01-16T04:00:00Z|     -13.012|48143-level-downs...|
|720215-level-stag...|2026-01-16T08:30:00Z|       0.392|720215-level-stag...|
|                NULL|                NULL|        NULL|                NULL|
|                NULL|                NULL|        NULL|        

Cast types

In [339]:
readings_df = readings_df.withColumn(
                "readingDatetime", col("readingDatetime").cast("timestamp")
                ).withColumn(
                "readingValue", col("readingValue").cast("float")
                )
readings_df.printSchema()

root
 |-- readingId: string (nullable = true)
 |-- readingDatetime: timestamp (nullable = true)
 |-- readingValue: float (nullable = true)
 |-- measurementId: string (nullable = true)



## Stations Data

In [280]:
stations_df = spark.read.format('json').load("raw_data/station_data/2026-01-11.json")

# --- Flatten the nested JSON structure ---
# 1. Select the 'items' array and ignore 'context' and 'metadata' fields
flattened_station_df = stations_df.select(explode("items").alias("item")) \
    .select(
        col("item.stationReference"),
        col("item.label").alias("stationLabel"),
        col("item.status"),
        col("item.town"),
        col("item.dateOpened").alias("dateOpened"),
        col("item.catchmentName"),
        col("item.riverName"),
        col("item.lat"),
        col("item.long")
        )

flattened_station_df.show()

+----------------+--------------------+--------------------+------------------+----------+--------------------+----------------+---------+---------+
|stationReference|        stationLabel|              status|              town|dateOpened|       catchmentName|       riverName|      lat|     long|
+----------------+--------------------+--------------------+------------------+----------+--------------------+----------------+---------+---------+
|          1029TH|     Bourton Dickler|http://environmen...| Little Rissington|1994-01-01|           Cotswolds|    River Dikler|51.874767|-1.740083|
|           E2043|     Surfleet Sluice|http://environmen...| Surfleet Seas End|1992-01-01|             Welland|      River Glen|52.845991|-0.100848|
|           52119|          Gaw Bridge|http://environmen...|Kingsbury Episcopi|1997-01-01|Parrett, Brue and...|   River Parrett|50.976043|-2.793549|
|          E21136|          Hemingford|http://environmen...|   Hemingford Grey|1996-10-01|Upper and Bedfor

Handle listed values (as they are malformed values)

In [281]:
for col_name in flattened_station_df.columns:
    malformed_count = flattened_station_df.filter(col(col_name).contains("[")).count()
    print(f"Malformed values in {col_name}: {malformed_count}")

Malformed values in stationReference: 0
Malformed values in stationLabel: 2
Malformed values in status: 2
Malformed values in town: 0
Malformed values in dateOpened: 1
Malformed values in catchmentName: 1
Malformed values in riverName: 0
Malformed values in lat: 1
Malformed values in long: 1


In [282]:
flattened_station_df.filter(col("stationLabel").contains("[")).show(truncate=False)

+----------------+---------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------+----------+----------------------------+---------------------------+-----------------+---------------------+----------------------+
|stationReference|stationLabel                                       |status                                                                                                                                               |town      |dateOpened                  |catchmentName              |riverName        |lat                  |long                  |
+----------------+---------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------+----------+----------------------------+---------------------------+--------

In [283]:
is_malformed = None # Set to None
for col_name in flattened_station_df.columns:
    condition = col(col_name).contains("[") # Check if the column contains '[' (indicating a string-list)
    is_malformed = condition if is_malformed is None else is_malformed | condition # Combine conditions with OR

quarantine_stations_df = flattened_station_df.filter(is_malformed)
clean_stations_df = flattened_station_df.filter(~is_malformed)

Check for null values

In [258]:
for col_name in clean_stations_df.columns:
    null_values = clean_stations_df.where(col(col_name).isNull()).count()
    print(f"Null_values in '{col_name}': {null_values}")

Null_values in 'stationReference': 0
Null_values in 'stationLabel': 0
Null_values in 'status': 0
Null_values in 'town': 0
Null_values in 'dateOpened': 0
Null_values in 'catchmentName': 0
Null_values in 'riverName': 0
Null_values in 'lat': 0
Null_values in 'long': 0


Transforming the status field

In [ ]:
# Extract status from the URL
transformed_stations_df = clean_stations_df.withColumn(
    "status", regexp_extract(col("status"), r'status([a-zA-Z]+)$', 1) # use PySpark regexp_extract function
)
transformed_stations_df.show()

+----------------+--------------------+------+------------------+----------+--------------------+----------------+---------+---------+
|stationReference|        stationLabel|status|              town|dateOpened|       catchmentName|       riverName|      lat|     long|
+----------------+--------------------+------+------------------+----------+--------------------+----------------+---------+---------+
|          1029TH|     Bourton Dickler|Active| Little Rissington|1994-01-01|           Cotswolds|    River Dikler|51.874767|-1.740083|
|           E2043|     Surfleet Sluice|Active| Surfleet Seas End|1992-01-01|             Welland|      River Glen|52.845991|-0.100848|
|           52119|          Gaw Bridge|Active|Kingsbury Episcopi|1997-01-01|Parrett, Brue and...|   River Parrett|50.976043|-2.793549|
|          E21136|          Hemingford|Active|   Hemingford Grey|1996-10-01|Upper and Bedford...|River Great Ouse|52.323618|-0.101287|
|            2067|             Swindon|Active|         

Cast the data types

In [286]:
flattened_station_df = flattened_station_df.withColumn(
                "dateOpened", col("dateOpened").cast("date")
                ).withColumn(
                "lat", col("lat").cast("float")
                ).withColumn(
                "long", col("long").cast("float")
                )
flattened_station_df.printSchema()

root
 |-- stationReference: string (nullable = true)
 |-- stationLabel: string (nullable = true)
 |-- status: string (nullable = true)
 |-- town: string (nullable = true)
 |-- dateOpened: date (nullable = true)
 |-- catchmentName: string (nullable = true)
 |-- riverName: string (nullable = true)
 |-- lat: float (nullable = true)
 |-- long: float (nullable = true)



# Spark SQL Analytical Demonstrations

In [382]:
# Set the DataFrames as temporary views
transformed_readings_df.createOrReplaceTempView("readings_df")
transformed_measurements_df.createOrReplaceTempView("measurements_df")

# Use Spark SQL to join the DataFrames and query relevant columns
transformed_df = spark.sql("""
                           SELECT readingDatetime, stationReference, parameter, readingValue, unitName
                           FROM readings_df
                           JOIN measurements_df
                           ON readings_df.measurementId = measurements_df.measurementId
                           SORT BY readingDateTime DESC
                           """)

transformed_df.show()

+--------------------+----------------+-----------+------------+--------+
|     readingDatetime|stationReference|  parameter|readingValue|unitName|
+--------------------+----------------+-----------+------------+--------+
|2026-01-16T08:41:40Z|           E4412|Water Level|       1.377|    mASD|
|2026-01-16T08:41:36Z|           E2920|Water Level|      22.642|    mAOD|
|2026-01-16T08:40:41Z|          E12290|Water Level|        48.8|    mAOD|
|2026-01-16T08:40:40Z|          E11600|Water Level|       52.28|    mAOD|
|2026-01-16T08:40:10Z|    DIFF_TB3_TB1|Water Level|       0.467|       m|
|2026-01-16T08:40:10Z|    DIFF_TB3_TB9|Water Level|       2.362|       m|
|2026-01-16T08:40:10Z|    DIFF_TB2_TB1|Water Level|       2.315|       m|
|2026-01-16T08:38:12Z|           E1959|Water Level|       3.201|    mAOD|
|2026-01-16T08:38:12Z|          E65301|Water Level|       3.201|    mAOD|
|2026-01-16T08:37:42Z|           E3300|Water Level|       5.161|    mAOD|
|2026-01-16T08:37:42Z|          E16141